# 🏭 Dry Granulation DOE: Material × Machine × Process Analysis

**Dataset:** Dry Granulation: Multi-Material Roll Compaction & Milling (Synthetic) v1.0  
**Publisher:** [Innovative Process Applications (IPA)](https://www.innovativeprocess.com)  
**License:** CC BY 4.0  
**Scientific basis:** Szappanos-Csordás (2018), Heinrich-Heine-Universität Düsseldorf

> ⚠️ This is **synthetic educational data** — not real measurements. See `README.md` for the full physical model.

---

This notebook demonstrates a multi-factor Quality-by-Design (QbD) workflow:

1. **Load & explore** the 720-run, 24-column dataset
2. **Compare materials** — plastic (MCC) vs. brittle (mannitol) vs. mixed
3. **Quantify machine design effects** — sealing system, roll surface, roll diameter
4. **Fit a response surface model** for ribbon density
5. **Define the design space** using the Zinchuk criterion (RD 0.60–0.80) + fines < 25%
6. **Build a classifier** — predict whether a run lands in the Zinchuk window

## Setup & Data Loading

In [ ]:
# Install dependencies (Colab has most of these pre-installed)
# !pip install -q pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, classification_report, ConfusionMatrixDisplay

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# === LOAD DATA ===
# Option A: If running locally or on Kaggle
# df = pd.read_csv('dry_granulation_multi_factor_v1.0.csv')

# Option B: If running on Google Colab, upload the CSV first:
# from google.colab import files
# uploaded = files.upload()  # Click 'Choose Files' and select the CSV
# df = pd.read_csv('dry_granulation_multi_factor_v1.0.csv')

# Option C: Load directly from GitHub (update URL after publishing)
# df = pd.read_csv('https://raw.githubusercontent.com/YOUR-ORG/ipa-datasets/main/ipa-granulation-dataset-v1.0/dry_granulation_multi_factor_v1.0.csv')

# For now, use whichever option works for your setup:
df = pd.read_csv('dry_granulation_multi_factor_v1.0.csv')

print(f'Dataset: {df.shape[0]} runs, {df.shape[1]} columns')
print(f'Materials: {df["material"].unique()}')
print(f'Sealing systems: {df["sealing_system"].unique()}')
print(f'Roll surfaces: {df["roll_surface"].unique()}')
df.head()

## Part 1: Material Deformation Behavior

A core concept from the granulation literature: **plastic materials** (MCC) are
dwell-time sensitive — roll speed strongly affects ribbon density. **Brittle
materials** (mannitol) fragment rapidly under pressure, so roll speed has
negligible effect on density.

Let’s verify this in our dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
materials = ['MCC_101', 'Mannitol_SD', 'MCC_Mannitol_Mix']
titles = ['MCC 101 (Plastic)', 'Mannitol (Brittle)', '1:1 Mixture']
colors = ['#008080', '#d4a017', '#2E86C1']  # IPA teal, gold, accent blue

for ax, mat, title, color in zip(axes, materials, titles, colors):
    subset = df[df['material'] == mat]
    ax.scatter(subset['roll_speed_rpm'], subset['ribbon_rel_density'],
              alpha=0.3, s=10, color=color)
    # Trend line
    z = np.polyfit(subset['roll_speed_rpm'], subset['ribbon_rel_density'], 1)
    x_line = np.linspace(1, 30, 100)
    ax.plot(x_line, np.polyval(z, x_line), color='black', linewidth=2)
    slope = z[0]
    ax.set_title(f'{title}\nSlope = {slope:.4f}')
    ax.set_xlabel('Roll Speed (rpm)')

axes[0].set_ylabel('Ribbon Relative Density')
fig.suptitle('Roll Speed Sensitivity by Material Deformation Type', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('- MCC (plastic): steep negative slope → strongly dwell-time sensitive')
print('- Mannitol (brittle): near-zero slope → RS has negligible effect')
print('- Mixture: intermediate behavior')

## Part 2: Machine Design Effects

Two key machine design choices from the literature:
- **Sealing system:** rim-rolls vs. side-sealing → affects density uniformity and fines
- **Roll surface:** knurled vs. smooth → affects nip angle and powder grip

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Density CV by sealing system
sns.boxplot(data=df, x='sealing_system', y='density_CV_percent',
            palette=['#cccccc', '#008080'], ax=axes[0])
axes[0].set_title('Density Uniformity (CV%) by Sealing')
axes[0].set_ylabel('Density CV %')
axes[0].set_xlabel('')

# Fines by sealing system
sns.boxplot(data=df, x='sealing_system', y='fines_fraction_pct',
            palette=['#cccccc', '#008080'], ax=axes[1])
axes[1].set_title('Fines Fraction by Sealing')
axes[1].set_ylabel('Fines < 100 \u00b5m (%)')
axes[1].set_xlabel('')

# Ribbon density by roll surface
sns.boxplot(data=df, x='roll_surface', y='ribbon_rel_density',
            palette=['#d4a017', '#1B2A3B'], ax=axes[2])
axes[2].set_title('Ribbon Density by Roll Surface')
axes[2].set_ylabel('Relative Density')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

print('Key findings:')
print(f'  Rim-rolls mean CV: {df[df.sealing_system=="rim_rolls"]["density_CV_percent"].mean():.1f}%')
print(f'  Side-sealing mean CV: {df[df.sealing_system=="side_sealing"]["density_CV_percent"].mean():.1f}%')
print(f'  Rim-rolls mean fines: {df[df.sealing_system=="rim_rolls"]["fines_fraction_pct"].mean():.1f}%')
print(f'  Side-sealing mean fines: {df[df.sealing_system=="side_sealing"]["fines_fraction_pct"].mean():.1f}%')

## Part 3: Densification Factor & Roll Diameter

The densification factor (DF = drawn-in width / gap width) increases with
roll diameter. This means **larger rolls achieve higher densification at the
same gap setting** — a fundamental scale-up consideration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for diam, color, marker in [(120, '#d4a017', 'o'), (250, '#008080', 's')]:
    subset = df[df['roll_diameter_mm'] == diam]
    axes[0].scatter(subset['gap_width_mm'], subset['densification_factor'],
                   alpha=0.3, s=10, color=color, marker=marker,
                   label=f'D = {diam} mm')

axes[0].set_xlabel('Gap Width (mm)')
axes[0].set_ylabel('Densification Factor')
axes[0].set_title('DF vs. Gap Width by Roll Diameter')
axes[0].legend()

# Ribbon density comparison
sns.boxplot(data=df, x='compactor_config', y='ribbon_rel_density',
            palette=['#d4a017', '#008080', '#2E86C1'], ax=axes[1])
axes[1].set_title('Ribbon Density by Compactor Configuration')
axes[1].set_ylabel('Relative Density')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Part 4: Response Surface Model for Ribbon Density

Fit a quadratic RSM model using the main process parameters plus categorical
encodings for material and machine design.

In [ ]:
# Encode categoricals
df_model = df.copy()
df_model['mat_code'] = LabelEncoder().fit_transform(df_model['material'])
df_model['seal_code'] = (df_model['sealing_system'] == 'rim_rolls').astype(int)
df_model['surf_code'] = (df_model['roll_surface'] == 'knurled').astype(int)

features = ['scf_kN_per_cm', 'gap_width_mm', 'roll_speed_rpm', 'feed_screw_rpm',
            'roll_diameter_mm', 'mat_code', 'seal_code', 'surf_code']

X = df_model[features].values
y = df_model['ribbon_rel_density'].values

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
Xp = poly.fit_transform(X)
model = LinearRegression().fit(Xp, y)
y_pred = model.predict(Xp)
r2 = r2_score(y, y_pred)

print(f'Quadratic RSM: R\u00b2 = {r2:.3f} ({Xp.shape[1]} features from {len(features)} inputs)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y, y_pred, alpha=0.2, s=8, color='#008080')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'k--')
axes[0].set_xlabel('Observed'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'RSM Fit (R\u00b2 = {r2:.3f})')

residuals = y - y_pred
axes[1].hist(residuals, bins=40, color='#008080', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Residuals (\u03c3 = {residuals.std():.4f})')
axes[1].axvline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

## Part 5: Design Space — Zinchuk Criterion + Fines Control

**QbD asks:** which combinations of material, machine, and process give us:
- Ribbon relative density in the Zinchuk window (0.60–0.80)
- Fines fraction < 25%
- Density CV < 5%

This is the multi-objective design space that pharmaceutical engineers must
navigate in real process development.

In [ ]:
# Define multi-objective spec
in_spec = (
    (df['ribbon_rel_density'] >= 0.60) &
    (df['ribbon_rel_density'] <= 0.80) &
    (df['fines_fraction_pct'] < 25.0) &
    (df['density_CV_percent'] < 5.0)
)
print(f'{in_spec.sum()} of {len(df)} runs meet ALL specs ({100*in_spec.mean():.1f}%)')
print()

# Breakdown by material and sealing
print('Spec compliance by Material \u00d7 Sealing System:')
ct = pd.crosstab(
    [df['material'], df['sealing_system']],
    in_spec,
    normalize='index'
).round(3) * 100
ct.columns = ['Out of Spec %', 'In Spec %']
print(ct)

# Design space scatter
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df.loc[~in_spec, 'scf_kN_per_cm'], df.loc[~in_spec, 'ribbon_rel_density'],
           alpha=0.2, s=8, color='lightgray', label='Out of spec')
scatter = ax.scatter(df.loc[in_spec, 'scf_kN_per_cm'], df.loc[in_spec, 'ribbon_rel_density'],
                     alpha=0.5, s=12, c=df.loc[in_spec, 'fines_fraction_pct'],
                     cmap='YlGn_r', label='In spec')
ax.axhline(0.60, color='red', linestyle='--', alpha=0.5, label='Zinchuk bounds')
ax.axhline(0.80, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Specific Compaction Force (kN/cm)')
ax.set_ylabel('Ribbon Relative Density')
ax.set_title('Design Space: Zinchuk Window + Fines < 25% + CV < 5%')
plt.colorbar(scatter, label='Fines %')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Part 6: Zinchuk Window Classifier

Can we predict whether a run will land in the optimal density window from
process inputs alone? This demonstrates how ML complements DOE in pharma
process development.

In [ ]:
# Binary target: does the run meet the Zinchuk criterion?
y_class = (df['in_zinchuk_window'] == 'Yes').astype(int).values

X_class = df_model[features].values
scaler = StandardScaler().fit(X_class)
X_scaled = scaler.transform(X_class)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf, X_scaled, y_class, cv=5, scoring='accuracy')
print(f'Random Forest 5-fold CV accuracy: {rf_scores.mean():.3f} \u00b1 {rf_scores.std():.3f}')

# Feature importance
rf.fit(X_scaled, y_class)
importances = pd.Series(rf.feature_importances_, index=features).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

importances.plot.barh(ax=axes[0], color='#008080')
axes[0].set_title('Feature Importance for Zinchuk Window Prediction')
axes[0].set_xlabel('Importance')

ConfusionMatrixDisplay.from_estimator(rf, X_scaled, y_class, ax=axes[1],
                                      cmap='Teals', display_labels=['Out', 'In'])
axes[1].set_title('Confusion Matrix (Training Set)')

plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_class, rf.predict(X_scaled),
                           target_names=['Out of Window', 'In Window']))

## Takeaways

This analysis demonstrates several key principles from roll compaction science:

1. **Material deformation type matters:** MCC (plastic) shows strong roll speed sensitivity;
   mannitol (brittle) does not. Process parameters must be tuned to the material.

2. **Machine design is not just plumbing:** rim-roll sealing systems produce meaningfully
   better density uniformity and fewer fines than side-sealing — a design choice that
   directly impacts downstream tablet quality.

3. **The densification factor links machine geometry to process outcome:** larger roll
   diameters achieve higher densification at the same gap setting, which is the
   fundamental scale-up relationship.

4. **QbD design spaces are multi-objective:** meeting the Zinchuk density criterion alone
   is insufficient — you must simultaneously control fines and uniformity.

5. **ML and DOE are complementary:** the RSM model captures the main trends; the
   Random Forest classifier handles the nonlinearities and interactions that a quadratic
   model misses.

---

For production-scale twin-feed-screw roller compactors that deliver uniform nip
feeding and consistent ribbon density, see IPA’s CL-series compactors:
**https://www.innovativeprocess.com**

*Dataset © 2026 Innovative Process Applications, CC BY 4.0. Synthetic
educational data — not real measurements.*

*Scientific basis: Szappanos-Csordás, K. (2018). Impact of material properties,
process parameters and roll compactor design on roll compaction. Doctoral
dissertation, Heinrich-Heine-Universität Düsseldorf.*